# SC4001 Phase 3, KIBA cold-target **s4004**: baseline against DANN at lambda 0.03, equal compute

Accelerator must be **GPU (T4 x2 or P100)**. `src/train_dann.py` refuses CPU by design.
Code and data come from the attached dataset `isaacwjk21/sc4001-project`; no internet or installs.

## Why this kernel exists

DAVIS now has a valid answer: with an adversary that demonstrably trains and compute held
equal, DANN does not improve cold-target generalisation. Paired by seed against a
patience-100 baseline, test MSE moved +0.0038 at `lambda_max` 0.03 (t(2) = +0.28), +0.0176 at
0.1 (t(2) = +1.19) and +0.0643 at 0.3 (t(2) = +1.89), none significant at three seeds, with CI
slightly worse in all three arms.

KIBA has since answered the same question on its s4001 partition, and the answer is
stronger than the DAVIS one: DANN is **worse on every seed in both metrics**, test MSE +0.0728
(t(2) = +8.92, p = 0.012) and test CI -0.0226 (t(2) = -5.12, p = 0.036), against a baseline of
0.3501 MSE. The adversary trained in all three runs, so that is a real cost and not a dead
gradient.

This kernel asks the one question that result leaves open: **is the deficit a property of the
method or of that single gene partition?** The four-way DAVIS spread showed the baseline itself
moves from 0.383 to 0.557 across partitions, and s4001 sits 0.104 below the four-partition
mean, so one partition is thin ground for any claim about KIBA. If DANN is also worse on s4004,
the Phase 3 conclusion is partition-independent on both benchmarks. If it is neutral or better
here, the s4001 deficit is partly a partition effect and must be reported as one.

## Design, and the three choices worth questioning

**80 epochs for both arms**, via `--max-epochs 80 --patience 80`, so there is no early stopping
and both arms get identical compute. DAVIS used 100 because its baseline peaked as late as
epoch 93. Every KIBA baseline measured so far peaked between epoch 28 and 74, so 80 covers the
latest observed peak with margin. This matters for cost as well as correctness: at the
KIBA-measured 1.49x DANN overhead, 100 epochs of KIBA DANN would run about 70 min and breach
the project's one-hour-per-run budget, while 80 epochs landed at 56.1 min on s4001.
Note that Ganin's schedule normalises progress by `max_epochs`, so lambda still saturates across
the budget and remains comparable to the DAVIS runs.

**One lambda, 0.03.** On DAVIS it produced the strongest adversary of the sweep (75% domain
accuracy, `domain_loss` down to 0.497 from ln 2 = 0.693) and the smallest regression penalty,
and lambda 0.3 destabilised two of three seeds. Since DAVIS showed the outcome barely moves
among lambdas whose adversary trains, spending three times the quota to re-establish that on
KIBA buys little. Worth noting against expectation: on KIBA s4001 this lambda produced a
*weaker* adversary than on DAVIS (domain accuracy 0.664-0.712 against 0.746-0.751) and still a
much larger penalty, so adversary strength does not predict the cost. The cost of this choice is that a null result here is open to "perhaps
another lambda would have worked"; that is accepted deliberately and should be stated in the
write-up rather than glossed.

**The baseline arm is re-run here**, unlike the DAVIS lambda sweep which reused a stored
baseline. The existing KIBA baselines were trained under `patience` 15 and stopped between
epoch 43 and 89, so they do not share this kernel's 80-epoch budget and cannot serve as an
equal-compute comparison. Their figures are printed at the end for context only.

`split_cold_target_s4004` only, three training seeds, six runs. This is the **hard end** of the two-partition bracket: KIBA baselines score 0.3682 on s4001 against 0.5331 on s4004, a +0.165 MSE penalty from the gene partition alone.

## Two safeguards

**Runs are interleaved by seed** (baseline, DANN, baseline, DANN, ...) rather than grouped by
arm. If the kernel is cut short by quota or the 12 h limit, what survives is complete
baseline/DANN pairs, which is how this comparison is read, instead of three baselines and no
DANN.

**An internal budget guard** stops launching new runs once elapsed wall time would not fit
another, projecting each run's cost from the slowest completed run of the same arm. The overhead is now
measured on KIBA rather than assumed from DAVIS: 42.08 s/epoch against 28.33 s/epoch, 1.49x, on
the s4001 grid. The guard stays because s4004 is a different partition and an underestimate
should truncate the grid cleanly rather than have the kernel killed with nothing printed.

## What the output answers

A **validity gate before any MSE**: per DANN run, the minimum `domain_loss` against
ln 2 = 0.693147, the maximum `domain_accuracy`, and a TRAINED or AT CHANCE verdict. Then best
validation MSE per run, test metrics at each run's own best checkpoint, and the
**paired-by-seed differences** with their per-seed signs, since three seeds is too few to read
a mean without them.

A deficit of similar size to s4001's makes the Phase 3 conclusion partition-independent. A
deficit much smaller, or absent, means the s4001 number is partly a property of that partition
and the KIBA claim has to be stated per partition rather than per dataset.

Expect roughly 4 h 45 min, from the s4001 grid's measured 37.8 min per baseline run and
56.1 min per DANN run. That grid took 281.6 min in total; s4004 is the same dataset and budget
on a different gene partition, so the cost should be close.

Outputs survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the entry point on the split being run before spending quota, so a loader or
# split-resolution fault costs seconds rather than an hour.
smoke = subprocess.run(
    [sys.executable, "src/train_dann.py", "--dataset", "kiba",
     "--split", "split_cold_target_s4004", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("Smoke run of src/train_dann.py failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# KIBA cold-target s4004, baseline against DANN at lambda 0.03, three seeds, equal compute.
#
# Budget: 80 epochs for BOTH arms (--max-epochs 80 --patience 80, so no early stopping).
# Every KIBA baseline measured so far peaked between epoch 28 and 74, so 80 covers the latest
# observed peak with margin. DAVIS needed 100 because its baseline peaked as late as 93; KIBA
# does not, and 100 epochs of KIBA DANN would breach the project's one-hour-per-run budget.
#
# Runs are INTERLEAVED by seed (baseline, dann, baseline, dann, ...) so that if the kernel is
# cut short by quota or the 12 h limit, what survives is complete baseline/DANN pairs rather
# than three baselines and no DANN. Paired-by-seed is how this comparison is read.
ROOT = "/kaggle/working/outputs/kiba-dann-s4004"
SPLIT = "split_cold_target_s4004"
LAMBDA = "0.03"
GRID = []
for seed in (4001, 4002, 4003):
    GRID.append(("src/train.py", "baseline", seed, []))
    GRID.append(("src/train_dann.py", "dann-lambda" + LAMBDA, seed,
                 ["--lambda-max", LAMBDA, "--discriminator-input", "protein"]))

# Internal budget guard. DANN overhead of 1.53x is measured on DAVIS, not KIBA, so the per-run
# cost here is a projection. Rather than risk the kernel being killed mid-run with nothing
# reported, stop launching new runs once the elapsed wall time would not fit another, and
# report what completed. 5.5 h of grid wall; the Kaggle kernel limit is 12 h.
BUDGET_SECONDS = 5.5 * 3600

started = time.time()
status = []
skipped = []
for script, arm, seed, extra in GRID:
    elapsed_so_far = time.time() - started
    # Project the next run's cost from the slowest completed run of the same arm, falling back
    # to the DAVIS-derived estimate before anything of that arm has finished.
    same_arm = [row[2] for row in status if row[0].startswith(arm)]
    projected = max(same_arm) if same_arm else (60 * 60 if "dann" in arm else 40 * 60)
    if elapsed_so_far + projected > BUDGET_SECONDS:
        skipped.append((arm, seed, projected))
        print("SKIPPING " + arm + " seed" + str(seed) + ": projected "
              + format(projected / 60, ".0f") + " min would exceed the "
              + format(BUDGET_SECONDS / 3600, ".1f") + " h grid budget ("
              + format(elapsed_so_far / 3600, ".2f") + " h used)", flush=True)
        continue
    t0 = time.time()
    label = arm + " kiba " + SPLIT + " seed" + str(seed)
    print()
    print("===== " + label + " (80 epochs, no early stop) =====", flush=True)
    run = subprocess.run(
        [sys.executable, script, "--dataset", "kiba", "--split", SPLIT,
         "--seed", str(seed), "--max-epochs", "80", "--patience", "80", "--resume"] + extra,
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT + "/" + arm), check=False,
    )
    wall = time.time() - t0
    status.append((label, run.returncode, wall))
    print("----- rc=" + str(run.returncode) + " wall=" + format(wall / 60, ".1f")
          + " min (" + format(wall / 80, ".1f") + " s/epoch) -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, wall in status:
    print("%-54s rc=%d %6.1f min  %5.1f s/epoch" % (label, rc, wall / 60, wall / 80))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")
print("SKIPPED FOR BUDGET:", skipped if skipped else "none")

# Measured DANN overhead on KIBA, which has only ever been assumed (1.53x from DAVIS).
base_rates = [w / 80 for l, rc, w in status if l.startswith("baseline") and rc == 0]
dann_rates = [w / 80 for l, rc, w in status if l.startswith("dann") and rc == 0]
if base_rates and dann_rates:
    import statistics
    print()
    print("KIBA DANN overhead: %.2f s/epoch vs baseline %.2f s/epoch = %.2fx"
          % (statistics.mean(dann_rates), statistics.mean(base_rates),
             statistics.mean(dann_rates) / statistics.mean(base_rates)))

# THE VALIDITY GATE, before any MSE. A DANN number means nothing if the discriminator never
# trained: at lambda_max 1.0 on DAVIS it sat at ln 2 for all 100 epochs.
LN2 = 0.6931471805599453
print()
print("ADVERSARY CHECK: domain_loss against ln 2 = %.6f" % LN2)
gate = []
for path in sorted(Path(ROOT).glob("**/history.csv")):
    arm = str(path).replace(ROOT + "/", "").split("/")[0]
    if not arm.startswith("dann"):
        continue
    rows = [line.split(",") for line in path.read_text().strip().splitlines()]
    header, body = rows[0], rows[1:]
    if not body:
        continue
    index = {name: position for position, name in enumerate(header)}
    if "domain_loss" not in index or "domain_accuracy" not in index:
        print("  skipped %s: no domain columns" % path)
        continue
    low = min(float(row[index["domain_loss"]]) for row in body)
    high = max(float(row[index["domain_accuracy"]]) for row in body)
    trained = (LN2 - low) > 0.005 or high > 0.55
    seed = int(str(path.parent).split("seed")[-1])
    gate.append(trained)
    print("  seed%d  min domain_loss %.6f  max domain_accuracy %.4f  %s"
          % (seed, low, high, "TRAINED" if trained else "AT CHANCE"))
print("%d of %d DANN runs had an adversary that departed from chance." % (sum(gate), len(gate)))
if gate and not any(gate):
    print("NO RUN HAD A WORKING ADVERSARY. The MSE numbers below do not measure DANN. "
          "Do not report them as a result about domain adaptation.")

print()
print("Best validation MSE over the 80-epoch budget:")
for path in sorted(Path(ROOT).glob("**/history.csv")):
    rows = [line.split(",") for line in path.read_text().strip().splitlines()]
    header, body = rows[0], rows[1:]
    if not body:
        continue
    index = {name: position for position, name in enumerate(header)}
    best = min(body, key=lambda row: float(row[index["val_mse"]]))
    arm = str(path).replace(ROOT + "/", "").split("/")[0]
    print("  %-18s %-12s best val_mse %.6f at epoch %-4s (final epoch %s)"
          % (arm, str(path.parent).split("/")[-1], float(best[index["val_mse"]]),
             best[index["epoch"]], body[-1][index["epoch"]]))

print()
print("Test metrics at each run's own best checkpoint.")
print("For reference, KIBA patience-15 baselines on this split: test MSE 0.5331 (sd 0.0418)")
print("  over 3 training seeds, CI 0.7318 (sd 0.0050). Those are NOT the comparison arm:")
print("  the baseline runs in this kernel are, because only they share the 80-epoch budget.")
pairs = {}
for path in sorted(Path(ROOT).glob("**/metrics.json")):
    payload = json.loads(path.read_text())
    arm = str(path).replace(ROOT + "/", "").split("/")[0]
    pairs.setdefault(payload["seed"], {})[arm] = payload
    print("  %-18s seed%d  test MSE %.4f  CI %.4f  val MSE %.4f  best_ep %d"
          % (arm, payload["seed"], payload["test"]["all"]["mse"],
             payload["test"]["all"]["ci"], payload["val"]["all"]["mse"],
             payload["best_epoch"]))

# Paired differences, which is how the comparison is read. Only seeds with both arms count.
complete = {seed: arms for seed, arms in pairs.items() if len(arms) == 2}
print()
print("%d seed(s) have both arms." % len(complete))
if complete:
    import statistics
    dm, dc = [], []
    for seed in sorted(complete):
        arms = complete[seed]
        base = next(v for k, v in arms.items() if k.startswith("baseline"))
        dann = next(v for k, v in arms.items() if k.startswith("dann"))
        d_mse = dann["test"]["all"]["mse"] - base["test"]["all"]["mse"]
        d_ci = dann["test"]["all"]["ci"] - base["test"]["all"]["ci"]
        dm.append(d_mse)
        dc.append(d_ci)
        print("  seed%d  dMSE %+.4f  dCI %+.4f" % (seed, d_mse, d_ci))
    print("  mean   dMSE %+.4f  dCI %+.4f" % (statistics.mean(dm), statistics.mean(dc)))
    if len(dm) > 1:
        print("  sd     dMSE  %.4f  dCI  %.4f" % (statistics.stdev(dm), statistics.stdev(dc)))
        print("  NOTE: with %d seeds, read the per-seed signs above before trusting the mean."
              % len(dm))


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
